In [ ]:
# === Mount Google Drive and install dependencies ===
from google.colab import drive
drive.mount("/content/drive")
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q

# 05 — C2: Feature Analysis

**Purpose:** Compute injection-sensitivity scores for all 6144 SAE features,
visualize the distribution, and characterize the top 50 injection-sensitive
features (positive and negative).

**Extends J3:** J3 inspected the top 20 features. C2 goes deeper — top 50 —
and provides a more thorough characterization.

**Prerequisites:**
- `checkpoints/sae_d6144_lambda1e-04.pt`
- `checkpoints/j2_activations.npz`
- `data/processed/iris_dataset_balanced.json`

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = '/content/drive/MyDrive/iris' if IN_COLAB else os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

In [ ]:
# === Load all artifacts ===
import torch
import numpy as np
import json
from pathlib import Path
from src.sae.architecture import SparseAutoencoder
from src.data.dataset import IrisDataset

dataset = IrisDataset.load('data/processed/iris_dataset_balanced.json')
texts = dataset.texts
labels = dataset.labels

data = np.load('checkpoints/j2_activations.npz')
activations = {i: data[f'layer_{i}'] for i in range(12)}

checkpoint = torch.load('checkpoints/sae_d6144_lambda1e-04.pt', map_location=device)
config = checkpoint['config']
sae = SparseAutoencoder(d_input=config['d_input'], expansion_factor=config['expansion_factor'],
                        sparsity_coeff=config.get('sparsity_coeff', 1e-4))
sae.load_state_dict(checkpoint['model_state_dict'])
sae = sae.to(device)
sae.eval()

TRAIN_LAYER = 0
print(f'Loaded: {len(texts)} examples, SAE with {sae.d_sae} features')

In [ ]:
# === Compute feature activations and sensitivity scores ===
from src.analysis.features import (compute_feature_activations,
                                    compute_sensitivity_scores,
                                    get_top_features,
                                    plot_sensitivity_distribution,
                                    plot_top_features_bar)

feature_matrix = compute_feature_activations(sae, activations[TRAIN_LAYER], device=device)
sensitivity = compute_sensitivity_scores(feature_matrix, np.array(labels))

In [ ]:
# === Sensitivity distribution ===
plot_sensitivity_distribution(sensitivity, save_path='results/figures/c2_sensitivity_distribution.png')

In [ ]:
# === Top 50 features ===
top_indices, top_values = get_top_features(sensitivity, k=50)

# Separate positive (injection) and negative (normal) features
pos_mask = top_values > 0
neg_mask = top_values < 0
print(f'Top 50 features: {pos_mask.sum()} injection-associated, {neg_mask.sum()} normal-associated')

plot_top_features_bar(top_indices[:20], top_values[:20],
                      save_path='results/figures/c2_top20_features.png')

In [ ]:
# === Feature dashboards for top 50 ===
from src.analysis.features import get_top_activating_examples, print_feature_dashboard

for feat_idx, sens_val in zip(top_indices, top_values):
    top_examples = get_top_activating_examples(
        feature_matrix, feat_idx, texts, labels, k=10
    )
    print_feature_dashboard(feat_idx, sens_val, top_examples)

In [ ]:
# === Coherence analysis for top 50 ===
coherence_results = []
for feat_idx, sens_val in zip(top_indices, top_values):
    top_examples = get_top_activating_examples(feature_matrix, feat_idx, texts, labels, k=10)
    expected_label = 1 if sens_val > 0 else 0
    matches = sum(1 for ex in top_examples if ex['label'] == expected_label)
    coherence_results.append({
        'feature': int(feat_idx), 'sensitivity': float(sens_val),
        'direction': 'injection' if sens_val > 0 else 'normal',
        'coherence': matches / 10, 'matches': matches,
    })

n_coherent = sum(1 for r in coherence_results if r['coherence'] >= 0.7)
mean_coherence = np.mean([r['coherence'] for r in coherence_results])

print(f'\nTop 50 features with >= 70% coherence: {n_coherent}/50')
print(f'Mean coherence: {mean_coherence:.0%}')

In [ ]:
# === Save C2 results ===
# Also save the sensitivity scores for use by C3 and C4
np.save('checkpoints/sensitivity_scores.npy', sensitivity)
np.save('checkpoints/feature_matrix.npy', feature_matrix)

c2_results = {
    'experiment': 'C2',
    'n_features_inspected': 50,
    'n_coherent_70pct': n_coherent,
    'mean_coherence': float(mean_coherence),
    'n_injection_features': int(pos_mask.sum()),
    'n_normal_features': int(neg_mask.sum()),
    'features': coherence_results,
}
with open('results/metrics/c2_feature_analysis.json', 'w') as f:
    json.dump(c2_results, f, indent=2)
print('Saved C2 results and feature matrix/sensitivity checkpoints')

## C2 Summary

Extended the J3 analysis from 20 to 50 features. Saved feature matrix and
sensitivity scores for use by C3 (detection comparison) and C4 (adversarial evasion).